In [ ]:
from models.CoreModel import CoreModel
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import cross_val_score
import pandas as pd

In [48]:
model = CoreModel()
model.load_data_from_csv("../data/ACME-HappinessSurvey2020.csv")

#use entire dataset
x = model.get_data()[["X1", "X2", "X3", "X4", "X5", "X6"]]
y = model.get_data()["Y"] 

In [49]:
models = {

    "Logistic Regression": {
        "k": 1,
        "pipeline": Pipeline([
            ("select", SelectKBest(score_func=f_classif, k=1)),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(random_state=42))
        ]),
        "params": {
            #"select__k": [1, 2, 3, 4, 5, 6],
            "model__C": [0.01, 0.1, 1, 10]
        }
    },

    "Decision Tree": {
        "k": 3,
        "pipeline": Pipeline([
            ("select", SelectKBest(score_func=f_classif, k=3)),
            ("model", DecisionTreeClassifier(random_state=42))
        ]),
        "params": {
            #"select__k": [1, 2, 3, 4, 5, 6],
            "model__max_depth": [1, 2, 3, 4, 6, 10],
            "model__min_samples_split": [2, 5, 7, 10],
            "model__min_samples_leaf": [1, 5, 7, 10]
        }
    },

    "XGBoost": {
        "k": 3,
        "pipeline": Pipeline([
            ("select", SelectKBest(score_func=f_classif, k=3)),
            ("model", XGBClassifier(random_state=42))
        ]),
        "params": {
            #"select__k": [1, 2, 3, 4, 5, 6],
            "model__max_depth": [1, 2, 3, 4, 6, 10],
            "model__n_estimators": [50, 100, 150, 200],
            "model__learning_rate": [0.05, 0.1, 0.2]
        }
    },
    "KNN": {
        "k": 3,
        "pipeline": Pipeline([
            ("select", SelectKBest(score_func=f_classif, k=3)),
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier())
        ]),
        "params": {
            #"select__k": [1, 2, 3, 4, 5, 6],
            "model__n_neighbors": [1, 2, 3, 5, 7, 9, 11]
        }
    }
}

In [50]:
cv = RepeatedStratifiedKFold(   
    n_splits=5,
    n_repeats=10,
    random_state=42
)

"""Note: StratifiedKFold was used to determine the best k number of features for each model 
        before using RepeatedStratifiedKFold to significantly reduce time to get results

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

"""

'Note: StratifiedKFold was used to determine the best k number of features for each model \n        before using RepeatedStratifiedKFold to significantly reduce time to get results\n\ncv = StratifiedKFold(\n    n_splits=5,\n    shuffle=True,\n    random_state=42\n)\n\n'

In [ ]:
results = []

def retrive_k_values(best_k = None):
        if best_k is not None:
            selector = SelectKBest(score_func=f_classif, k=best_k)
            selector.fit(x, y)
            return list(x.columns[selector.get_support()])
        else:
            return "N/A"

for name, config in models.items():
    # Grid search
    grid_search = GridSearchCV(config["pipeline"], config["params"], cv=cv, scoring="accuracy", n_jobs=-1)

    grid_search.fit(x, y)

    #==== code to find what the best k values are for each model ====
    #best_k = grid_search.best_params_.get("select__k", None)

    best_k = config["k"]
    selected_features = retrive_k_values(best_k)

    # Get best model
    best_model = grid_search.best_estimator_

    # Evaluate best model using RepeatedStratifiedKFold
    scores = cross_val_score(best_model, x, y, cv=cv, scoring="accuracy")
    

    results.append({
        "Model": name,
        "Best K": best_k,
        "Selected Features": selected_features,
        "Best Accuracy": scores.mean(),
        "Best Standard Deviation": scores.std(),
        "Best Parameters": grid_search.best_params_
        
    })


In [ ]:
results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

              Model  Best K Selected Features  Best Accuracy  Best Standard Deviation                                                                       Best Parameters
Logistic Regression       1              [X1]       0.621477                 0.080607                                                                      {'model__C': 10}
      Decision Tree       3      [X1, X5, X6]       0.654062                 0.078715 {'model__max_depth': 4, 'model__min_samples_leaf': 10, 'model__min_samples_split': 2}
            XGBoost       3      [X1, X5, X6]       0.664523                 0.080654      {'model__learning_rate': 0.2, 'model__max_depth': 4, 'model__n_estimators': 150}
                KNN       3      [X1, X5, X6]       0.671662                 0.082538                                                             {'model__n_neighbors': 3}
